# Experiment 5: GRU with Word Embedding<br>
## Objective:<br>
To evaluate the performance of a Bidirectional GRU model for sentiment classification using word embeddings.<br>
<br>
## Setup: <br>
- Feature: Embedding
- Model: BiGRU
- Dataset: Amazon Reviews

## Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
print(os.getcwd())

c:\Users\USER\Desktop\FYP\fyp-nlp-reviews\experiments


In [5]:
from importlib import reload
import src.data.preprocess as p
reload(p)
from src.data.preprocess import preprocess_text

## Load Data

In [7]:
df = pd.read_csv("../data/processed/cleaned.csv")

print(df.shape)
df.head()

(57994, 12)


,reviewId,reviewDate,mainDepartment,subDepartment,productName,reviewTitle,reviewStar,reviewText,inconsistentStatus,sentiment,review_length,clean_text
0,R1TWIP923TUPRV,"Reviewed in the United States on August 8, 2021",Beauty and Personal Care,"Foot,Hand & Nail Care",Hard As Hoof Nail Strengthening Cream with Coc...,Didn't work,1.0,No change in nails at all.,0,0.0,26,no change in nails at all
1,R1N5I12682L3YY,"Reviewed in the United States on August 2, 2021",Beauty and Personal Care,"Foot,Hand & Nail Care",Hard As Hoof Nail Strengthening Cream with Coc...,Worst nail product I have ever used.,1.0,Garbage! Do not waste your $$$. My nails were ...,1,0.0,157,garbage do not waste your my nails were worse ...
2,R1S3UGVIPYX6W5,"Reviewed in the United States on July 27, 2021",Beauty and Personal Care,"Foot,Hand & Nail Care",Hard As Hoof Nail Strengthening Cream with Coc...,Just didn't work,1.0,Nails are still brittle,1,0.0,23,nails are still brittle
3,R3RELGNNBTGZ2V,"Reviewed in the United States on July 26, 2021",Beauty and Personal Care,"Foot,Hand & Nail Care",Hard As Hoof Nail Strengthening Cream with Coc...,Scum,1.0,Didn’t do any improvement at all.,0,0.0,33,didn’t do any improvement at all
4,RLQ9C5R2KYV67,"Reviewed in the United States on July 24, 2021",Beauty and Personal Care,"Foot,Hand & Nail Care",Hard As Hoof Nail Strengthening Cream with Coc...,Okay for cuticles,1.0,"Helped cuticles, didn't do much for my nails. ...",0,0.0,129,helped cuticles didnt do much for my nails i g...


## Prepare Features & Labels

In [12]:
X = df["clean_text"].fillna("").astype(str)
y = df["sentiment"]

## Train/Test Split

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Tokenization & Padding 

In [14]:
max_words = 20000
max_len = 300

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

## Model

In [15]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128),
    Bidirectional(GRU(128)),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

## Compile

In [16]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## Train

In [17]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 550s 938ms/step - accuracy: 0.8109 - loss: 0.4133 - val_accuracy: 0.8589 - val_loss: 0.3268
Epoch 2/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 481s 829ms/step - accuracy: 0.8947 - loss: 0.2646 - val_accuracy: 0.8699 - val_loss: 0.3163
Epoch 3/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 445s 768ms/step - accuracy: 0.9223 - loss: 0.2040 - val_accuracy: 0.8707 - val_loss: 0.3109
Epoch 4/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 428s 738ms/step - accuracy: 0.9391 - loss: 0.1634 - val_accuracy: 0.8684 - val_loss: 0.3556
Epoch 5/10
580/580 ━━━━━━━━━━━━━━━━━━━━ 428s 737ms/step - accuracy: 0.9507 - loss: 0.1334 - val_accuracy: 0.8570 - val_loss: 0.4042


## Evaluate

In [18]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("GRU Test Accuracy:", accuracy)

363/363 ━━━━━━━━━━━━━━━━━━━━ 29s 80ms/step - accuracy: 0.8691 - loss: 0.3166
GRU Test Accuracy: 0.8691266775131226
